In [1]:
# Performance config
import os

CPU_THREADS = min(32, os.cpu_count() or 32)
os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(CPU_THREADS)
os.environ["NUMEXPR_NUM_THREADS"] = str(CPU_THREADS)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(CPU_THREADS)
os.environ["TOKENIZERS_PARALLELISM"] = "true"

REQUIRE_CUDA = True  # set False for CPU-only notebooks

print(f"CPU threads set to: {CPU_THREADS}")

try:
    import torch
except Exception as e:
    torch = None
    if REQUIRE_CUDA:
        raise RuntimeError("CUDA required but torch is not available.") from e

if torch is not None:
    torch.set_num_threads(CPU_THREADS)
    torch.set_num_interop_threads(min(4, CPU_THREADS))
    if REQUIRE_CUDA and not torch.cuda.is_available():
        raise RuntimeError("CUDA required but not available.")
    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = True
        print("CUDA device:", torch.cuda.get_device_name(0))
    else:
        print("CUDA not available; running on CPU.")


CPU threads set to: 12
CUDA device: NVIDIA GeForce RTX 5070 Ti


# Kvasir-VQA x1 — BLIP-2 VQA fine-tuning

Fine-tune a stronger VQA transformer (BLIP-2 / InstructBLIP) on the x1 splits.

Outputs saved to `2_modeling/06_blip2_finetune/results/`.
Heavy artifacts (checkpoints + final model) go to `2_modeling/06_blip2_finetune/out/`.


In [2]:
import os
# Avoid TF/Keras import issues in transformers (this notebook uses PyTorch)
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["ACCELERATE_MIXED_PRECISION"] = "no"

from pathlib import Path
import json
import random

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset

from transformers import Blip2Processor, Blip2ForConditionalGeneration, BlipProcessor, BlipForQuestionAnswering
from transformers import TrainingArguments, Trainer
import evaluate


In [3]:
# Paths & config

def find_kvasir_x1_root() -> Path:
    import os
    env_root = os.environ.get("KVASIR_VQA_X1_ROOT")
    if env_root:
        p = Path(env_root).expanduser().resolve()
        if (p / "0_dataset_prep").exists():
            return p
        raise RuntimeError(f"KVASIR_VQA_X1_ROOT set but missing 0_dataset_prep: {p}")

    if "__file__" in globals():
        p = Path(__file__).resolve()
        root = p.parents[2]
        if root.name == "Kvasir_VQA_x1" and (root / "0_dataset_prep").exists():
            return root

    cwd = Path.cwd().resolve()
    for p in [cwd] + list(cwd.parents):
        if p.name == "Kvasir_VQA_x1" and (p / "0_dataset_prep").exists():
            return p

    raise RuntimeError(
        "Could not locate Kvasir_VQA_x1 dataset root. 
"
        "Run this notebook from within the Kvasir_VQA_x1 folder, 
"
        "or set KVASIR_VQA_X1_ROOT."
    )

DATA_ROOT = find_kvasir_x1_root()
META_CSV = DATA_ROOT / "0_dataset_prep" / "out" / "metadata" / "metadata_enriched.csv"
RESULTS_DIR = DATA_ROOT / "2_modeling" / "06_blip2_finetune" / "results"
OUT_DIR = DATA_ROOT / "2_modeling" / "06_blip2_finetune" / "out"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Swap to another backbone if you prefer (e.g., 'Salesforce/instructblip-vicuna-7b' for instruction-tuned VQA)

# Memory/precision helpers
LOW_VRAM = True  # set True on consumer GPUs to reduce VRAM use
FORCE_FP16 = True  # set True to avoid bf16 CUBLAS issues
FREEZE_VISION = True   # only applies to BLIP2
FREEZE_QFORMER = True  # only applies to BLIP2
FREEZE_LM = False      # only applies to BLIP2 (set True to freeze language model too)
MODEL_FAMILY = "blip"  # "blip" (smaller) or "blip2"
if MODEL_FAMILY == "blip":
    MODEL_NAME = "Salesforce/blip-vqa-base"
    PROMPT_TEMPLATE = "{question}"
    USE_8BIT = False
    DEVICE_MAP = None
    TORCH_DTYPE = torch.float32
    GRADIENT_CHECKPOINTING = False
else:
    MODEL_NAME = "Salesforce/blip2-flan-t5-xl"
    PROMPT_TEMPLATE = "Question: {question}
Answer:"
    USE_8BIT = True  # set False if bitsandbytes not available
    DEVICE_MAP = "auto"  # use HF accelerate device placement; set None to disable
    USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    TORCH_DTYPE = torch.bfloat16 if USE_BF16 else (torch.float16 if torch.cuda.is_available() else torch.float32)
    GRADIENT_CHECKPOINTING = True

if MODEL_FAMILY == "blip2" and LOW_VRAM:
    # Typical BLIP2 low-VRAM setup: train Q-Former only
    FREEZE_VISION = True
    FREEZE_QFORMER = False
    FREEZE_LM = True
# AMP (fp16/bf16) flags are set later based on actual model dtype

FORCE_CPU = False  # set True to force CPU
USE_GPU = torch.cuda.is_available() and not FORCE_CPU
if USE_GPU:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

# If running BLIP on a smaller GPU, enable mixed precision + checkpointing to reduce VRAM
if USE_GPU and MODEL_FAMILY == "blip" and LOW_VRAM:
    USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    TORCH_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
    GRADIENT_CHECKPOINTING = True

# Force fp16 if bf16 kernels are unstable on this GPU/driver
if USE_GPU and FORCE_FP16:
    USE_BF16 = False
    TORCH_DTYPE = torch.float16


USE_FP16_TRAIN = USE_GPU and TORCH_DTYPE == torch.float16
USE_BF16_TRAIN = USE_GPU and TORCH_DTYPE == torch.bfloat16

DEVICE = torch.device("cuda" if USE_GPU else "cpu")
SEED = 42
MAX_ANSWER_LEN = 16
QUESTION_MAX_LEN = 64
MAX_GEN_TOKENS = 12


if LOW_VRAM:
    # Shorter sequences reduce attention memory
    QUESTION_MAX_LEN = min(QUESTION_MAX_LEN, 32)
    MAX_ANSWER_LEN = min(MAX_ANSWER_LEN, 8)
    if MODEL_FAMILY == "blip2":
        MAX_GEN_TOKENS = min(MAX_GEN_TOKENS, 8)

BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
NUM_EPOCHS = 1
LR = 1e-5

MAX_TRAIN_SAMPLES = None  # set int for smoke tests
MAX_VAL_SAMPLES = None
MAX_TEST_SAMPLES = None


torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print("Data root:", DATA_ROOT)
print("Metadata:", META_CSV)
print("Results dir:", RESULTS_DIR)
print("Out dir:", OUT_DIR)
print("Device:", DEVICE)


Data root: /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1
Metadata: /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/0_dataset_prep/out/metadata/metadata_enriched.csv
Out dir: /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/2_modeling/06_blip2_finetune/out
Device: cuda


In [4]:
# Load metadata
meta = pd.read_csv(META_CSV)

images_base = DATA_ROOT / "0_dataset_prep"
meta["image_path"] = meta["image_path"].apply(
    lambda p: str((images_base / p).resolve()) if not Path(p).is_absolute() else p
)

if "split" not in meta.columns:
    raise RuntimeError("Missing 'split' column. Run dataset prep split step first.")

train_df = meta[meta["split"] == "train"].reset_index(drop=True)
val_df = meta[meta["split"] == "validation"].reset_index(drop=True)
test_df = meta[meta["split"] == "test"].reset_index(drop=True)

# Drop rows missing answers
train_df = train_df.dropna(subset=["answer"]).reset_index(drop=True)
val_df = val_df.dropna(subset=["answer"]).reset_index(drop=True)
test_df = test_df.dropna(subset=["answer"]).reset_index(drop=True)

print({"train": len(train_df), "val": len(val_df), "test": len(test_df)})


{'train': 143594, 'val': 0, 'test': 15955}


In [5]:
# Optional subsampling
if MAX_TRAIN_SAMPLES is not None:
    train_df = train_df.sample(min(MAX_TRAIN_SAMPLES, len(train_df)), random_state=SEED).reset_index(drop=True)
if MAX_VAL_SAMPLES is not None:
    val_df = val_df.sample(min(MAX_VAL_SAMPLES, len(val_df)), random_state=SEED).reset_index(drop=True)
if MAX_TEST_SAMPLES is not None:
    test_df = test_df.sample(min(MAX_TEST_SAMPLES, len(test_df)), random_state=SEED).reset_index(drop=True)

print({"train": len(train_df), "val": len(val_df), "test": len(test_df)})

{'train': 143594, 'val': 0, 'test': 15955}


In [6]:
# Load processor + model
if MODEL_FAMILY == "blip2":
    processor = Blip2Processor.from_pretrained(MODEL_NAME)

    bnb_config = None
    if USE_8BIT:
        try:
            from transformers import BitsAndBytesConfig
            bnb_config = BitsAndBytesConfig(load_in_8bit=True)
        except Exception as e:
            print("8-bit quantization unavailable, falling back to full precision:", e)

    if bnb_config is not None:
        try:
            model = Blip2ForConditionalGeneration.from_pretrained(
                MODEL_NAME,
                quantization_config=bnb_config,
                device_map=DEVICE_MAP,
            )
        except Exception as e:
            print("8-bit load failed, falling back to full precision:", e)
            bnb_config = None

    if bnb_config is None:
        model = Blip2ForConditionalGeneration.from_pretrained(
            MODEL_NAME,
            torch_dtype=TORCH_DTYPE,
            device_map=DEVICE_MAP if DEVICE_MAP is not None else None,
        )
        if DEVICE_MAP is None:
            model.to(DEVICE)
else:
    processor = BlipProcessor.from_pretrained(MODEL_NAME)
    model = BlipForQuestionAnswering.from_pretrained(MODEL_NAME)
    if TORCH_DTYPE in (torch.float16, torch.bfloat16):
        model = model.to(DEVICE, dtype=TORCH_DTYPE)
    else:
        model = model.to(DEVICE)

if GRADIENT_CHECKPOINTING and hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
processor.tokenizer.padding_side = "right"
if hasattr(model.config, "text_config"):
    model.config.text_config.pad_token_id = processor.tokenizer.pad_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
if hasattr(model.config, "use_cache"):
    model.config.use_cache = False

def _freeze_module(mod):
    for p in mod.parameters():
        p.requires_grad = False

# Optional freezing to reduce VRAM for BLIP2
if MODEL_FAMILY == "blip2":
    if FREEZE_VISION and hasattr(model, "vision_model"):
        _freeze_module(model.vision_model)
    if FREEZE_QFORMER and hasattr(model, "qformer"):
        _freeze_module(model.qformer)
    if FREEZE_LM and hasattr(model, "language_model"):
        _freeze_module(model.language_model)

# Report trainable params
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {all_params:,} ({trainable/all_params:.2%})")
if trainable == 0:
    raise RuntimeError("No trainable parameters. Adjust FREEZE_* flags or LOW_VRAM settings.")



Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Trainable params: 384,672,572 / 384,672,572 (100.00%)


In [7]:
class VQADataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")
        prompt = PROMPT_TEMPLATE.format(question=str(row["question"]))
        inputs = processor(images=image, text=prompt, return_tensors="pt", padding="max_length", truncation=True, max_length=QUESTION_MAX_LEN)
        labels = processor.tokenizer(
            str(row["answer"]) if pd.notna(row["answer"]) else "",
            max_length=MAX_ANSWER_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        ).input_ids.squeeze(0)

        # BLIP uses labels as decoder_input_ids when none are provided. Avoid -100 there.
        if MODEL_FAMILY == "blip":
            decoder_input_ids = labels.clone()
            decoder_attention_mask = (decoder_input_ids != processor.tokenizer.pad_token_id).long()
            labels = labels.clone()
            labels[labels == processor.tokenizer.pad_token_id] = -100
        else:
            labels[labels == processor.tokenizer.pad_token_id] = -100

        item = {k: v.squeeze(0) for k, v in inputs.items()}
        item["labels"] = labels
        if MODEL_FAMILY == "blip":
            item["decoder_input_ids"] = decoder_input_ids
            item["decoder_attention_mask"] = decoder_attention_mask
        return item

def collate_fn(batch):
    keys = batch[0].keys()
    return {k: torch.stack([b[k] for b in batch]) for k in keys}

def _check_vocab(item, tokenizer, name="sample"):
    vocab = getattr(tokenizer, "vocab_size", None)
    if vocab is None:
        return
    if "input_ids" in item:
        ids = item["input_ids"]
        if int(ids.max()) >= vocab or int(ids.min()) < 0:
            raise RuntimeError(f"{name}: input_ids out of range for vocab_size={vocab}")
    if "decoder_input_ids" in item:
        dec_ids = item["decoder_input_ids"]
        if int(dec_ids.max()) >= vocab or int(dec_ids.min()) < 0:
            raise RuntimeError(f"{name}: decoder_input_ids out of range for vocab_size={vocab}")
    if "labels" in item:
        labels = item["labels"]
        bad = (labels >= vocab) | ((labels < 0) & (labels != -100))
        if bad.any():
            raise RuntimeError(f"{name}: labels out of range for vocab_size={vocab}")

train_ds = VQADataset(train_df)
val_ds = VQADataset(val_df)
test_ds = VQADataset(test_df)

print("Sample prompt:", PROMPT_TEMPLATE.format(question=train_df.iloc[0]["question"]))

# Sanity check vocab alignment
_sample = train_ds[0]
_check_vocab(_sample, processor.tokenizer, name="train_ds[0]")





Sample prompt: Are there any abnormalities, polyps, or anatomical landmarks visible in the image?


In [8]:
class VQATrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        # Drop HF's helper arg that T5 models don't accept
        inputs = dict(inputs)
        inputs.pop("num_items_in_batch", None)
        labels = inputs.pop("labels", None)
        outputs = model(**inputs, labels=labels)
        loss = outputs.loss if hasattr(outputs, "loss") else outputs["loss"]
        return (loss, outputs) if return_outputs else loss

training_args = TrainingArguments(
    output_dir=str(OUT_DIR / "checkpoints"),
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LR,
    use_cpu=not USE_GPU,
    fp16=USE_FP16_TRAIN,
    bf16=USE_BF16_TRAIN,
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_steps=200,
    save_total_limit=2,
    remove_unused_columns=False,
    report_to=[],
)

trainer = VQATrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collate_fn,
)

print(trainer)


In [9]:
# Train
trainer.train()

# Save final model
final_dir = OUT_DIR / "final_model"
final_dir.mkdir(parents=True, exist_ok=True)
trainer.save_model(final_dir)
processor.save_pretrained(final_dir)
print("Saved model to", final_dir)


ValueError: Attempting to unscale FP16 gradients.

In [ ]:
# Evaluate with BLEU/ROUGE on test split
bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")

model.eval()

def generate_answer(row):
    img = Image.open(row["image_path"]).convert("RGB")
    prompt = PROMPT_TEMPLATE.format(question=str(row["question"]))
    inputs = processor(images=img, text=prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_GEN_TOKENS)
    return processor.tokenizer.decode(out[0], skip_special_tokens=True).strip()

preds = []
refs = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="BLIP-2 eval"):
    preds.append(generate_answer(row))
    refs.append(str(row["answer"]))

bleu_refs = [[r] for r in refs]
bleu_score = bleu.compute(predictions=preds, references=bleu_refs)["bleu"]
rouge_l = rouge.compute(predictions=preds, references=refs)["rougeL"]

results = {
    "bleu": bleu_score,
    "rougeL": rouge_l,
    "model": MODEL_NAME,
    "split": "test",
    "num_examples": len(test_df),
}
metrics_path = RESULTS_DIR / "metrics.json"
with open(metrics_path, "w") as f:
    json.dump(results, f, indent=2)

pred_path = RESULTS_DIR / "predictions.jsonl"
out_df = test_df.copy()
out_df["pred"] = preds
out_df.to_json(pred_path, orient="records", lines=True)

print("Wrote", metrics_path)
print("Wrote", pred_path)
